# 🔐 Cipher Classifier — Модуль 3: Интерфейс Gradio

В этом ноутбуке мы:
1. Загружаем обученную модель из Модуля 2
2. Реализуем валидацию загружаемого файла
3. Запускаем веб-интерфейс на Gradio

## 📦 Импорты

In [1]:
import sys
!{sys.executable} -m pip install gradio==3.50.2 gradio_client==0.6.1

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import gradio as gr
import joblib
import numpy as np
import string
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # для корректной работы внутри Gradio

print('✅ Все библиотеки загружены')
print(f'   Gradio версия: {gr.__version__}')

/Users/denisrezcik/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Все библиотеки загружены
   Gradio версия: 3.50.2


## 📂 Загрузка модели и признаков

In [3]:
model = joblib.load('cipher_model.pkl')
FEATURE_NAMES = joblib.load('feature_names.pkl')

CIPHER_LABELS = model.classes_
print(f'✅ Модель загружена')
print(f'   Классы: {list(CIPHER_LABELS)}')

✅ Модель загружена
   Классы: ['atbash', 'bacon', 'caesar', 'plaintext', 'playfair', 'rail_fence', 'vigenere']


## 🧮 Функции извлечения признаков

Копируем те же функции что и в Модуле 2 — они нужны для обработки нового текста.

In [14]:
COMMON_BIGRAMS = ['TH','HE','IN','ER','AN','RE','ON','EN','AT','OU']

def letter_frequencies(text):
    text = ''.join(c for c in text.upper() if c.isalpha())
    total = len(text) if len(text) > 0 else 1
    return [text.count(chr(ord('A') + i)) / total for i in range(26)]

def index_of_coincidence(text):
    text = ''.join(c for c in text.upper() if c.isalpha())
    N = len(text)
    if N < 2: return 0.0
    counts = [text.count(chr(ord('A') + i)) for i in range(26)]
    return sum(n * (n - 1) for n in counts) / (N * (N - 1))

def shannon_entropy(text):
    text = ''.join(c for c in text.upper() if c.isalpha())
    total = len(text)
    if total == 0: return 0.0
    probs = [text.count(chr(ord('A') + i)) / total for i in range(26)]
    return -sum(p * np.log2(p) for p in probs if p > 0)

def unique_letters(text):
    return len(set(c for c in text.upper() if c.isalpha()))

def bigram_features(text):
    text = ''.join(c for c in text.upper() if c.isalpha())
    total = len(text) - 1
    if total <= 0: return [0.0] * len(COMMON_BIGRAMS)
    bigrams = [text[i:i+2] for i in range(total)]
    return [bigrams.count(bg) / total for bg in COMMON_BIGRAMS]

def double_letter_ratio(text):
    text = ''.join(c for c in text.upper() if c.isalpha())
    if len(text) < 2: return 0.0
    doubles = sum(1 for i in range(len(text) - 1) if text[i] == text[i+1])
    return doubles / (len(text) - 1)

def extract_features(text):
    return (
        letter_frequencies(text) +
        [index_of_coincidence(text)] +
        [shannon_entropy(text)] +
        [unique_letters(text)] +
        bigram_features(text) +
        [double_letter_ratio(text)]
    )

print('✅ Функции признаков загружены')

✅ Функции признаков загружены


## ✅ Валидация входного текста

Проверяем два условия:
- Текст содержит только латинские буквы (пробелы допускаются)
- Длина текста не менее 20 символов (иначе частотный анализ ненадёжен)

In [15]:
MIN_LENGTH = 20

def validate_text(text: str) -> tuple:
    """
    Проверяет текст перед анализом.
    Возвращает (is_valid: bool, error_message: str)
    """
    letters_only = ''.join(c for c in text if c.isalpha())
    
    if len(letters_only) == 0:
        return False, '❌ Файл пустой или не содержит букв.'
    
    non_latin = [c for c in letters_only if not c.isascii()]
    if non_latin:
        return False, f'❌ Найдены не-латинские символы: {", ".join(set(non_latin))}. Все шифры работают только с латинским алфавитом.'
    
    if len(letters_only) < MIN_LENGTH:
        return False, f'❌ Текст слишком короткий ({len(letters_only)} букв). Минимум — {MIN_LENGTH} символов для надёжного анализа.'
    
    return True, ''

# Тест валидации
test_inputs = [
    ('KHOOR ZRUOG KHOOR ZRUOG KHOOR', 'корректный'),
    ('Hi',                             'слишком короткий'),
    ('Привет мир',                     'не латиница'),
    ('',                               'пустой'),
]
for text, desc in test_inputs:
    valid, msg = validate_text(text)
    print(f'{desc:20s}: valid={valid}  {msg if msg else "OK"}')

корректный          : valid=True  OK
слишком короткий    : valid=False  ❌ Текст слишком короткий (2 букв). Минимум — 20 символов для надёжного анализа.
не латиница         : valid=False  ❌ Найдены не-латинские символы: в, П, е, р, и, м, т. Все шифры работают только с латинским алфавитом.
пустой              : valid=False  ❌ Файл пустой или не содержит букв.


## 📈 Функция построения графика частот

In [16]:
def plot_frequency(text: str) -> plt.Figure:
    """Строит график частотного анализа букв для переданного текста"""
    text_clean = ''.join(c for c in text.upper() if c.isalpha())
    total = len(text_clean) if len(text_clean) > 0 else 1
    freqs = [text_clean.count(chr(ord('A') + i)) / total for i in range(26)]
    letters = list(string.ascii_uppercase)

    fig, ax = plt.subplots(figsize=(10, 3))
    bars = ax.bar(letters, freqs, color='#3498db', alpha=0.8, edgecolor='white', linewidth=0.5)

    # Подсвечиваем самые частые буквы
    max_freq = max(freqs)
    for bar, freq in zip(bars, freqs):
        if freq >= max_freq * 0.8:
            bar.set_color('#e74c3c')

    ax.set_title('Частотный анализ букв', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Буква')
    ax.set_ylabel('Относительная частота')
    ax.set_facecolor('#f8f9fa')
    fig.patch.set_facecolor('white')
    plt.tight_layout()
    return fig

print('✅ Функция графика готова')

✅ Функция графика готова


## 🔍 Описания шифров

Текст который покажем пользователю после предсказания.

In [17]:
CIPHER_INFO = {
    'plaintext': (
        '📄 Открытый текст (Plaintext)',
        'Текст не зашифрован. Распределение букв соответствует типичному английскому языку '
        '(самые частые: E, T, A, O, I). Индекс совпадений IC ≈ 0.065.'
    ),
    'caesar': (
        '🔑 Шифр Цезаря',
        'Моноалфавитный шифр сдвига. Каждая буква сдвигается на фиксированное число позиций: '
        'E(x) = (x + k) mod 26. Форма распределения букв сохраняется — просто сдвигается. '
        'Индекс совпадений IC ≈ 0.065 — такой же как у plaintext.'
    ),
    'vigenere': (
        '🔐 Шифр Виженера',
        'Многоалфавитный шифр. Каждая буква шифруется своим сдвигом по ключевому слову: '
        'E(xᵢ) = (xᵢ + k_{i mod m}) mod 26. Распределение букв более равномерное, '
        'IC ≈ 0.045 — заметно ниже чем у Цезаря.'
    ),
    'rail_fence': (
        '🚂 Шифр Rail Fence',
        'Шифр перестановки. Буквы записываются зигзагом по нескольким «рельсам» и читаются построчно. '
        'Распределение букв не меняется (это не замена, а перестановка), '
        'поэтому IC такой же как у plaintext.'
    ),
    'atbash': (
        '🪞 Шифр Атбаш',
        'Зеркальная замена букв: A↔Z, B↔Y, ... Формально: E(x) = 25 - x. '
        'Распределение букв зеркально отражается — редкие буквы английского '
        '(X, Z) становятся частыми (A, B) и наоборот.'
    ),
    'playfair': (
        '🔲 Шифр Плейфера',
        'Шифрует пары букв (диграммы) с помощью матрицы 5×5. '
        'Ключевая особенность: в зашифрованном тексте никогда не встречаются удвоенные буквы (AA, BB...). '
        'Буквы I и J объединяются в одну клетку матрицы.'
    ),
    'bacon': (
        '🥓 Шифр Бэкона',
        'Бинарное кодирование: каждая буква заменяется на 5-символьную последовательность из A и B. '
        'Например: A=AAAAA, B=AAAAB, Z=BBBBB. '
        'В тексте присутствуют только две буквы — это самый надёжный признак для распознавания.'
    ),
}

print('✅ Описания шифров загружены')

✅ Описания шифров загружены


## ⚙️ Основная функция обработки

In [18]:
def analyze_file(file) -> tuple:
    """
    Главная функция интерфейса.
    Принимает загруженный .txt файл, возвращает результаты анализа.
    """
    # --- Чтение файла ---
    if file is None:
        return '⚠️ Файл не загружен.', '', '', None

    try:
        with open(file.name, 'r', encoding='utf-8') as f:
            text = f.read().strip()
    except Exception as e:
        return f'❌ Ошибка чтения файла: {e}', '', '', None

    # --- Валидация ---
    is_valid, error_msg = validate_text(text)
    if not is_valid:
        return error_msg, '', '', None

    # --- Извлечение признаков и предсказание ---
    features = np.array([extract_features(text)])
    proba = model.predict_proba(features)[0]
    predicted = model.classes_[np.argmax(proba)]
    confidence = np.max(proba) * 100

    # --- Результат ---
    cipher_title, cipher_desc = CIPHER_INFO[predicted]

    result_text = f"{cipher_title}\n\nУверенность модели: {confidence:.1f}%"

    # Шкала вероятностей
    proba_lines = ['Вероятности по всем классам:\n']
    for cls, prob in sorted(zip(model.classes_, proba), key=lambda x: -x[1]):
        bar = '█' * int(prob * 30)
        proba_lines.append(f"{cls:12s} {bar} {prob*100:.1f}%")
    proba_text = '\n'.join(proba_lines)

    # Статистика текста
    letters_only = ''.join(c for c in text.upper() if c.isalpha())
    ic = index_of_coincidence(text)
    entropy = shannon_entropy(text)
    uniq = unique_letters(text)
    stats_text = (
        f"📊 Статистика текста\n\n"
        f"Длина (букв):           {len(letters_only)}\n"
        f"Уникальных букв:        {uniq} из 26\n"
        f"Индекс совпадений IC:   {ic:.4f}\n"
        f"Энтропия Шеннона H:     {entropy:.4f} бит\n"
        f"Удвоенных пар букв:     {double_letter_ratio(text):.4f}\n\n"
        f"💡 О шифре:\n{cipher_desc}"
    )

    # График
    fig = plot_frequency(text)

    return result_text, proba_text, stats_text, fig

print('✅ Основная функция готова')

✅ Основная функция готова


## 🚀 Запуск интерфейса Gradio

In [20]:
with gr.Blocks(title='Cipher Classifier', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🔐 Cipher Classifier
    **ML-классификатор криптографических шифров**

    Загрузи `.txt` файл с зашифрованным текстом — модель определит тип шифра.
    Поддерживаются: Цезарь, Виженер, Rail Fence, Атбаш, Плейфер, Бэкон, открытый текст.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(
                label='📄 Загрузи .txt файл',
                file_types=['.txt']
            )
            analyze_btn = gr.Button(
                '⚡ Определить шифр',
                variant='primary',
                size='lg'
            )

        with gr.Column(scale=2):
            result_output = gr.Textbox(
                label='🎯 Результат',
                lines=3,
                interactive=False
            )
            proba_output = gr.Textbox(
                label='📊 Вероятности по классам',
                lines=9,
                interactive=False
            )

    with gr.Row():
        freq_plot = gr.Plot(label='📈 Частотный анализ букв')

    with gr.Row():
        stats_output = gr.Textbox(
            label='🔬 Статистика и описание шифра',
            lines=10,
            interactive=False
        )

    analyze_btn.click(
        fn=analyze_file,
        inputs=[file_input],
        outputs=[result_output, proba_output, stats_output, freq_plot]
    )

    gr.Markdown("""
    ---
    *Модель: Random Forest · 40 признаков · 7 классов · Точность 91.7%*
    """)

demo.launch()

Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------


## ✅ Итог модуля 3

После запуска последней ячейки Gradio откроет интерфейс по адресу `http://127.0.0.1:7860` в браузере.

**Что умеет интерфейс:**
- Загрузка `.txt` файла
- Валидация: только латиница, минимум 20 букв
- Предсказание шифра с уверенностью
- Шкала вероятностей по всем 7 классам
- График частотного анализа
- Статистика текста (IC, энтропия, ...)
- Описание определённого шифра с формулой

**Проект завершён! 🎉**